# NDCG Evaluation for ELSA + SAE POI Recommender

This notebook evaluates the NDCG (Normalized Discounted Cumulative Gain) metrics for the trained ELSA + SAE models trained on Yelp POI data.

**Model Details:**
- ELSA: 512-dimensional latent space autoencoder
- SAE: TopK SAE (2048-dim, k=32)
- Dataset: CA state filtered with k-core=5 filtering
- Test split: 1558 users × 2742 items

In [5]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.models.collaborative_filtering import ELSA, recall_at_k, ndcg_at_k
from src.models.sparse_autoencoder import TopKSAE
from src.models.sae_cf_model import ELSASAEModel
from src.data.preprocessing import build_csr
from src.data.yelp_loader import load_reviews
from src.utils import CheckpointManager

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Path to checkpoint from most recent training run
checkpoint_dir = Path("../outputs/20260317_211058/checkpoints")
output_dir = checkpoint_dir.parent
results_file = output_dir / "training_results.json"

# Load training results with config
with open(results_file) as f:
    results = json.load(f)

config = results["config"]
data_shapes = results["data_shapes"]

print(f"✓ Loaded training results from {output_dir}")
print(f"\nData Summary:")
print(f"  - Train users: {data_shapes['train_users']}")
print(f"  - Val users: {data_shapes['val_users']}")
print(f"  - Test users: {data_shapes['test_users']}")
print(f"  - Items: {data_shapes['n_items']}")
print(f"  - Latent dim: {data_shapes['latent_dim']}")

# Device
device = torch.device("cpu")  # Using CPU for evaluation
print(f"\nUsing device: {device}")

✓ Loaded training results from ..\outputs\20260317_211058

Data Summary:
  - Train users: 5606
  - Val users: 623
  - Test users: 1558
  - Items: 2742
  - Latent dim: 512

Using device: cpu


In [8]:
# Fix paths to use absolute paths
# Yelp-JSON is in parent directory of the repo
cwd = Path.cwd()
project_root = cwd.parent.parent.parent  # Go up from notebooks -> Diplomov-pr-ce -> Github -> Diplomka
yelp_dir = project_root / "Yelp-JSON"

parquet_dir_abs = yelp_dir / "yelp_parquet"
db_path_abs = yelp_dir / "yelp.duckdb"

print(f"Corrected paths:")
print(f"  Parquet dir: {parquet_dir_abs}")
print(f"  DuckDB path: {db_path_abs}")
print(f"  Exists: parquet={parquet_dir_abs.exists()}, db={db_path_abs.exists()}")

# Update config
config["data"]["parquet_dir"] = str(parquet_dir_abs)
config["data"]["db_path"] = str(db_path_abs)

Corrected paths:
  Parquet dir: c:\Users\elisk\Desktop\2024-25\Diplomka\Yelp-JSON\yelp_parquet
  DuckDB path: c:\Users\elisk\Desktop\2024-25\Diplomka\Yelp-JSON\yelp.duckdb
  Exists: parquet=True, db=True


In [3]:
# Initialize checkpoint manager
checkpoint_mgr = CheckpointManager(checkpoint_dir)

# Load ELSA model
elsa = ELSA(
    n_items=data_shapes["n_items"],
    latent_dim=config["elsa"]["latent_dim"]
).to(device)

elsa_info = checkpoint_mgr.load(elsa, checkpoint_name="elsa_best", device=device)
print(f"✓ Loaded ELSA model (epoch {elsa_info['epoch']})")

# Load SAE model
sae_name = f"sae_r{config['sae']['width_ratio']}_k{config['sae']['k']}_best"
sae = TopKSAE(
    input_dim=config["elsa"]["latent_dim"],
    hidden_dim=config["sae"]["width_ratio"] * config["elsa"]["latent_dim"],
    k=config["sae"]["k"],
    l1_coef=config["sae"]["l1_coef"]
).to(device)

sae_info = checkpoint_mgr.load(sae, checkpoint_name=sae_name, device=device)
print(f"✓ Loaded SAE model (epoch {sae_info['epoch']})")

# Create combined model
model = ELSASAEModel(
    n_items=data_shapes["n_items"],
    latent_dim=config["elsa"]["latent_dim"],
    sae_hidden_dim=config["sae"]["width_ratio"] * config["elsa"]["latent_dim"],
    k=config["sae"]["k"],
    l1_coef=config["sae"]["l1_coef"]
).to(device)

# Copy weights
model.elsa.load_state_dict(elsa.state_dict())
model.sae.load_state_dict(sae.state_dict())
model.eval()

print(f"✓ Created combined ELSA+SAE model")

✓ Loaded ELSA model (epoch 24)
✓ Loaded SAE model (epoch 49)
✓ Created combined ELSA+SAE model


In [11]:
# Load Yelp data with CA state filter
from src.data.yelp_loader import load_businesses

parquet_dir = Path(config["data"]["parquet_dir"])
db_path = Path(config["data"]["db_path"])

print(f"Loading businesses from {parquet_dir}...")
print(f"  - State filter: {config['data']['state_filter']}")

# First, load businesses filtered by state
businesses = load_businesses(
    parquet_dir,
    db_path=db_path,
    state_filter=config["data"]["state_filter"]
)
print(f"✓ Loaded {len(businesses)} businesses from {config['data']['state_filter']}")

# Get business IDs for this state
business_ids = set(businesses['business_id'].values)

# Load reviews for these businesses
print(f"\nLoading reviews for {len(business_ids)} businesses...")
reviews = load_reviews(
    parquet_dir,
    db_path=db_path,
    business_ids=business_ids,
    pos_threshold=config["data"]["pos_threshold"]
)
print(f"✓ Loaded {len(reviews)} reviews")

# Apply k-core filtering
min_reviews_per_user = config['data']['min_review_count']
user_interactions = reviews.groupby('user_id')['business_id'].count()
valid_users = user_interactions[user_interactions >= min_reviews_per_user].index
reviews_filtered = reviews[reviews['user_id'].isin(valid_users)]
print(f"✓ Filtered to {len(reviews_filtered)} reviews from {len(valid_users)} users (min {min_reviews_per_user} reviews each)")

# Build CSR matrix
print("Building CSR matrix...")
dataset = build_csr(reviews_filtered)
X_csr = dataset.csr
print(f"✓ Built CSR matrix: {X_csr.shape[0]} users × {X_csr.shape[1]} items")

# Split into train/test
n_users = X_csr.shape[0]
user_indices = np.arange(n_users)
train_users, test_users = train_test_split(
    user_indices,
    test_size=1 - config["data"]["train_test_split"],
    random_state=config["data"]["seed"]
)

# Get test set
X_test_csr = X_csr[test_users]
gt_mask = X_test_csr.toarray().astype(bool)
X_test = torch.tensor(gt_mask, dtype=torch.float32)

print(f"\n✓ Test set: {X_test.shape[0]} users × {X_test.shape[1]} items")
print(f"  - Sparsity: {(1 - X_test.count_nonzero().item() / (X_test.shape[0] * X_test.shape[1])):.4f}")

Loading businesses from c:\Users\elisk\Desktop\2024-25\Diplomka\Yelp-JSON\yelp_parquet...
  - State filter: CA
✓ Loaded 5203 businesses from CA

Loading reviews for 5203 businesses...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded 253750 reviews
✓ Filtered to 94490 reviews from 8717 users (min 5 reviews each)
Building CSR matrix...
✓ Built CSR matrix: 8717 users × 4853 items

✓ Test set: 1744 users × 4853 items
  - Sparsity: 0.9979


In [ ]:
# Generate predictions for test set
# Note: Truncate test set to match training item size (2742 items)
n_items_train = data_shapes["n_items"]

if X_test.shape[1] > n_items_train:
    print(f"Warning: Test set has {X_test.shape[1]} items, but model was trained with {n_items_train}")
    print(f"  - Truncating to first {n_items_train} items")
    X_test_aligned = X_test[:, :n_items_train]
else:
    X_test_aligned = X_test

batch_size = 256
all_scores_elsa = []
all_scores_sae = []

print("Computing recommendations...")
with torch.no_grad():
    for i in range(0, X_test_aligned.shape[0], batch_size):
        batch = X_test_aligned[i:i+batch_size].to(device)
        
        # Get ELSA reconstruction scores
        scores_elsa_batch = model.elsa(batch).detach().cpu().numpy()
        all_scores_elsa.append(scores_elsa_batch)
        
        # Get ELSA latent representation
        latent = model.elsa.encode(batch)
        
        # Get SAE reconstruction scores (forward returns tuple: recon, h_sparse, h_pre)
        sae_recon, _, _ = model.sae(latent)
        scores_sae_batch = sae_recon.detach().cpu().numpy()
        all_scores_sae.append(scores_sae_batch)

# Concatenate predictions
scores_elsa = np.vstack(all_scores_elsa)
scores_sae = np.vstack(all_scores_sae)

print(f"✓ Generated predictions")
print(f"  - ELSA scores shape: {scores_elsa.shape}")
print(f"  - SAE scores shape: {scores_sae.shape}")

# Update ground truth mask to match truncated test set
gt_mask = X_test_aligned.numpy().astype(bool)

  - Truncating to first 2742 items
Computing recommendations...


AttributeError: 'tuple' object has no attribute 'detach'

In [ ]:
# Compute NDCG and Recall at different K values
k_values = [5, 10, 20, 50, 100]

def compute_metrics(scores, gt_mask, k_values):
    """Compute Recall@K and NDCG@K metrics."""
    metrics = {}
    
    for k in k_values:
        # Get top-k predictions
        top_indices = np.argsort(-scores, axis=1)[:, :k]
        
        # Recall@k
        recalls = []
        for i in range(scores.shape[0]):
            y_true = gt_mask[i]
            y_pred = top_indices[i]
            if y_true.sum() > 0:
                hit = y_true[y_pred].sum()
                recall = hit / min(k, y_true.sum())
                recalls.append(recall)
        
        # NDCG@k
        ndcgs = []
        for i in range(scores.shape[0]):
            y_true = gt_mask[i]
            y_pred = top_indices[i]
            if y_true.sum() > 0:
                ndcg = ndcg_at_k(y_true, y_pred, k=k)
                ndcgs.append(ndcg)
        
        metrics[f'Recall@{k}'] = np.mean(recalls) if recalls else 0.0
        metrics[f'NDCG@{k}'] = np.mean(ndcgs) if ndcgs else 0.0
    
    return metrics

# Compute metrics for ELSA
print("Computing NDCG for ELSA...")
metrics_elsa = compute_metrics(scores_elsa, gt_mask, k_values)

# Compute metrics for SAE
print("Computing NDCG for SAE...")
metrics_sae = compute_metrics(scores_sae, gt_mask, k_values)

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)

# Create comparison dataframe
results_df = pd.DataFrame({
    'ELSA': metrics_elsa,
    'SAE': metrics_sae
}).round(4)

print("\n" + results_df.to_string())

In [ ]:
# Prepare data for visualization
ndcg_elsa = [metrics_elsa[f'NDCG@{k}'] for k in k_values]
ndcg_sae = [metrics_sae[f'NDCG@{k}'] for k in k_values]
recall_elsa = [metrics_elsa[f'Recall@{k}'] for k in k_values]
recall_sae = [metrics_sae[f'Recall@{k}'] for k in k_values]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NDCG plot
axes[0].plot(k_values, ndcg_elsa, 'o-', label='ELSA', linewidth=2, markersize=8)
axes[0].plot(k_values, ndcg_sae, 's-', label='SAE', linewidth=2, markersize=8)
axes[0].set_xlabel('K (top-k items)', fontsize=12)
axes[0].set_ylabel('NDCG Score', fontsize=12)
axes[0].set_title('NDCG@K Comparison', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(k_values)

# Recall plot
axes[1].plot(k_values, recall_elsa, 'o-', label='ELSA', linewidth=2, markersize=8)
axes[1].plot(k_values, recall_sae, 's-', label='SAE', linewidth=2, markersize=8)
axes[1].set_xlabel('K (top-k items)', fontsize=12)
axes[1].set_ylabel('Recall Score', fontsize=12)
axes[1].set_title('Recall@K Comparison', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(k_values)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete")

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print(f"\nELSA Performance:")
print(f"  - NDCG@10: {metrics_elsa['NDCG@10']:.4f}")
print(f"  - NDCG@50: {metrics_elsa['NDCG@50']:.4f}")
print(f"  - Recall@10: {metrics_elsa['Recall@10']:.4f}")
print(f"  - Recall@50: {metrics_elsa['Recall@50']:.4f}")

print(f"\nSAE Performance:")
print(f"  - NDCG@10: {metrics_sae['NDCG@10']:.4f}")
print(f"  - NDCG@50: {metrics_sae['NDCG@50']:.4f}")
print(f"  - Recall@10: {metrics_sae['Recall@10']:.4f}")
print(f"  - Recall@50: {metrics_sae['Recall@50']:.4f}")

# Save results to JSON
eval_results = {
    "test_users": int(X_test.shape[0]),
    "test_items": int(X_test.shape[1]),
    "elsa_metrics": {k: float(v) for k, v in metrics_elsa.items()},
    "sae_metrics": {k: float(v) for k, v in metrics_sae.items()}
}

output_file = Path("../outputs/20260317_211058/ndcg_evaluation_results.json")
with open(output_file, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✓ Results saved to {output_file}")

## 6. Visualize NDCG Results

## 5. Calculate NDCG Scores

## 4. Generate Model Predictions

## 3. Prepare Test Data

## 2. Load Configuration and Models

## 1. Import Required Libraries